In [ ]:
# 필요한 라이브러리 임포트
import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks, butter, filtfilt
import time
from collections import deque
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq

# MediaPipe 얼굴 메쉬 초기화
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True)

def display_triangular_regions(frame, regions):
    """
    화면에 삼각형 영역(ROI)을 표시하는 함수.
    각 ROI의 꼭짓점 좌표를 기반으로 삼각형을 그려준다.
    """
    for triangle_points in regions:
        # 삼각형을 프레임에 그린다.
        cv2.polylines(frame, [triangle_points], isClosed=True, color=(0, 255, 0), thickness=2)

# 실시간 BPM 계산을 위한 데이터 구조 초기화
bpm_values = deque(maxlen=10 * 30)  # 약 10초 동안의 데이터 저장 (30 FPS 기준)
bpm_values_1 = deque(maxlen=5 * 30)  # 약 5초 동안의 데이터 저장 (30 FPS 기준)

def extract_rgb_signals_by_face_regions(frame, landmarks):
    """
    얼굴의 ROI(Region of Interest)를 정의하고, 각 영역에서 G(Green) 채널의 신호를 추출하는 함수.
    """
    h, w, _ = frame.shape  # 프레임의 높이와 너비 추출

    # 각 영역을 정의하는 랜드마크 인덱스
    # 이마 부위는 제거됨
    left_cheek_1 = [118, 119, 101]
    left_cheek_2 = [187, 50, 205]
    left_cheek_3 = [207, 214, 212]
    left_cheek_4 = [203, 165, 206]
    left_cheek_5 = [100, 36, 142]
    left_cheek_6 = [117, 111, 123]
    
    right_cheek_1 = [347, 348, 330]
    right_cheek_2 = [280, 425, 411]
    right_cheek_3 = [427, 432, 434]
    right_cheek_4 = [423, 426, 391]
    right_cheek_5 = [329, 371, 266]
    right_cheek_6 = [346, 340, 352]
    
    left_chin_indices = [194, 211, 32, 201]
    mid_chin_indices = [200, 83, 313]
    right_chin_indices = [418, 431, 262, 421]

    # ROI 영역 정의
    roi_landmarks = [
        left_cheek_1, left_cheek_2, left_cheek_3, left_cheek_4, left_cheek_5, left_cheek_6,
        right_cheek_1, right_cheek_2, right_cheek_3, right_cheek_4, right_cheek_5, right_cheek_6,
        left_chin_indices, mid_chin_indices, right_chin_indices
    ]

    mean_rgbs = []  # 각 영역에서 추출된 G 신호의 평균값을 저장
    regions = []  # 각 영역의 꼭짓점 좌표를 저장

    for indices in roi_landmarks:
        # 각 삼각형 꼭짓점 좌표 계산
        triangle_points = np.array([[int(landmarks[idx].x * w), int(landmarks[idx].y * h)] for idx in indices])

        # 빈 마스크 생성
        mask = np.zeros((h, w), dtype=np.uint8)
        # 삼각형 영역을 마스크에 채움
        cv2.fillPoly(mask, [triangle_points], 255)

        # 원본 프레임에서 삼각형 영역 추출
        masked_roi = cv2.bitwise_and(frame, frame, mask=mask)

        # G 채널의 평균값 계산
        mean_rgb = cv2.mean(masked_roi, mask=mask)[1]  # G 채널 값
        mean_rgbs.append(mean_rgb)

        # 삼각형 영역의 꼭짓점 저장
        regions.append(triangle_points)

    # 각 영역의 평균 G 신호와 삼각형 영역 좌표 반환
    return np.array(mean_rgbs), regions

def pos_algorithm(S):
    """
    POS 알고리즘을 적용하여 RGB 신호의 변화를 계산.
    - 입력 S: ROI에서 추출된 RGB 신호 배열.
    """
    S = np.array(S)
    mean_g = np.mean(S)  # G 채널의 평균값 계산
    C = (S / mean_g) - 1  # G 신호의 변화율 계산
    return C  # G 신호 변화율 반환

def bandpass_filter(signal, lowcut=0.7, highcut=3.0, fs=60.0, order=7):
    """
    대역 통과 필터를 사용하여 신호를 필터링.
    - 입력 signal: 필터링할 신호.
    - lowcut, highcut: 대역 통과 필터의 낮은/높은 주파수 경계.
    - fs: 샘플링 주파수.
    - order: 필터의 차수.
    """
    nyquist = 0.5 * fs  # 나이퀴스트 주파수 계산
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')  # 필터 설계
    return filtfilt(b, a, signal)  # 필터 적용

def calculate_heart_rate(signal, fps):
    """
    신호에서 심박수를 계산하는 함수.
    - 입력 signal: 신호 데이터.
    - fps: 초당 프레임 수.
    """
    peaks, _ = find_peaks(signal, distance=fps * 0.4)  # peak 찾기
    peak_intervals = np.diff(peaks) / fps  # peak 간의 간격 계산
    return 60.0 / np.median(peak_intervals) if len(peak_intervals) > 0 else 0

def calculate_heart_rate_fft(signal, fps):
    """
    FFT를 사용하여 심박수를 계산하는 함수.
    - 입력 signal: 신호 데이터.
    - fps: 초당 프레임 수.
    """
    # 신호 필터링
    filtered_signal = bandpass_filter(signal, lowcut=0.7, highcut=3.0, fs=fps, order=7)
    
    # FFT 수행
    N = len(filtered_signal)
    yf = fft(filtered_signal)
    xf = fftfreq(N, 1 / fps)

    # 양의 주파수 성분만 사용
    positive_freqs = xf[:N // 2]
    positive_magnitudes = np.abs(yf[:N // 2])

    # 심박수 범위 내의 주파수 필터링
    valid_idx = (positive_freqs >= 0.7) & (positive_freqs <= 3.0)
    filtered_freqs = positive_freqs[valid_idx]
    filtered_magnitudes = positive_magnitudes[valid_idx]

    # 최대 에너지 주파수 찾기
    if len(filtered_magnitudes) > 0:
        peak_freq = filtered_freqs[np.argmax(filtered_magnitudes)]
        peak_bpm = peak_freq * 60  # bpm 변환
    else:
        peak_bpm = 0

    return peak_bpm

def moving_average(data, window_size=5):
    """
    이동 평균 계산 함수.
    """
    data = list(data)
    return np.mean(data) if len(data) < window_size else np.mean(data[-window_size:])

def fig2data(fig):
    """
    Matplotlib Figure를 NumPy 배열로 변환하는 함수.
    """
    fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    buf = buf.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    return buf

def main():
    cap = cv2.VideoCapture(0)  # 웹캠에서 영상 스트림을 캡처
    fps = cap.get(cv2.CAP_PROP_FPS) or 30  # FPS(Frame Per Second)를 가져오거나 기본값 30으로 설정
    rgb_signals_history = [deque(maxlen=300) for _ in range(15)]  # 각 ROI(Region of Interest)에 대해 RGB 신호를 저장할 13개의 deque 생성
    heart_rate_history = deque(maxlen=5)  # 최근 5개의 심박수를 저장할 deque
    last_heart_rate = None  # 마지막으로 계산된 심박수
    last_display_time = time.time()  # 마지막으로 디스플레이를 업데이트한 시간
    update_interval = 2  # 디스플레이를 업데이트하는 간격(초)

    # matplotlib 그래프 설정
    plt.switch_backend('agg')  # 그래프를 화면에 표시하지 않고 메모리에서 렌더링
    fig, ax = plt.subplots(figsize=(6, 4))  # 6x4 크기의 플롯 생성
    line, = ax.plot([], [])  # 초기화된 빈 플롯 라인
    plt.close()  # 인터페이스에 그래프 표시를 끔

    pos_signal_history = deque(maxlen=300)  # POS 알고리즘에서 추출된 신호를 저장할 deque
    amplification_factor = 50  # 시각화를 위한 신호 증폭 계수

    while True:
        ret, frame = cap.read()  # 웹캠에서 프레임을 읽어옴
        if not ret:  # 프레임을 읽지 못한 경우
            continue  # 루프를 다시 시작 (break 대신 continue를 사용해 종료되지 않음)
        frame = cv2.flip(frame, 1)  # 프레임을 좌우 반전하여 거울 효과 생성

        # MediaPipe를 사용하여 얼굴 랜드마크 탐지
        results = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))  # RGB로 변환 후 얼굴 랜드마크 탐지
        if results.multi_face_landmarks:  # 얼굴 랜드마크가 감지된 경우
            face_landmarks = results.multi_face_landmarks[0].landmark  # 첫 번째 얼굴의 랜드마크 정보

            # 얼굴에서 RGB 신호를 추출
            rgb_signals, regions = extract_rgb_signals_by_face_regions(frame, face_landmarks)  # ROI 별 RGB 신호 및 해당 영역 추출

            # 각 구역(ROI)별 심박수 계산
            heart_rates = []
            for i in range(15):  # 13개의 ROI를 모두 처리
                rgb_signals_history[i].append(rgb_signals[i])  # 각 ROI의 RGB 신호를 해당 deque에 추가

                # 신호 데이터가 충분히 수집되었을 때만 분석 진행 (5초 동안의 데이터)
                if len(rgb_signals_history[i]) > int(fps * 5):
                    pos_signal = pos_algorithm(rgb_signals_history[i])  # POS 알고리즘 적용
                    filtered_signal = bandpass_filter(pos_signal, fs=fps)  # 신호를 대역 통과 필터로 필터링
                    heart_rate = calculate_heart_rate(filtered_signal, fps)  # 필터링된 신호에서 심박수 계산
                    heart_rates.append(heart_rate)  # 계산된 심박수를 리스트에 추가

                    # 첫 번째 ROI에 대해 시각화를 추가적으로 수행
                    if i == 0:
                        # 신호를 표준화 (평균 0, 표준편차 1)
                        standardized_signal = (filtered_signal - np.mean(filtered_signal)) / np.std(filtered_signal)
                        # 신호를 증폭하여 시각적으로 보기 쉽게 만듦
                        amplified_signal = standardized_signal[-1] * amplification_factor
                        pos_signal_history.append(amplified_signal)

                        # 그래프 데이터 업데이트
                        xdata = np.arange(len(pos_signal_history))  # x축 데이터
                        ydata = np.array(pos_signal_history)  # y축 데이터
                        line.set_data(xdata, ydata)  # 플롯 데이터 설정
                        ax.set_xlim(0, max(100, len(pos_signal_history)))  # x축 범위 설정
                        ax.set_ylim(np.min(ydata) - 1, np.max(ydata) + 1)  # y축 범위 설정

                        # 그래프 이미지를 OpenCV로 변환 및 표시
                        plot_img = fig2data(fig)  # matplotlib 그래프를 이미지 데이터로 변환
                        plot_img = cv2.cvtColor(plot_img, cv2.COLOR_RGB2BGR)  # 이미지를 RGB에서 BGR로 변환
                        cv2.imshow('POS Signal', plot_img)  # OpenCV 창에 그래프 표시

            # ROI에서 추출한 심박수 데이터가 충분히 있을 때 심박수 계산
            if len(heart_rates) >= 3:
                median_hr = np.median(heart_rates)  # ROI에서 계산된 심박수들의 중앙값 계산
                differences = np.abs(heart_rates - median_hr)  # 각 심박수와 중앙값 간의 차이 계산
                top_indices = np.argsort(differences)[:3]  # 중앙값에 가장 가까운 상위 3개의 인덱스 선택
                top_3_heart_rates = np.array(heart_rates)[top_indices]  # 선택된 상위 3개의 심박수
                final_heart_rate = np.mean(top_3_heart_rates)  # 상위 3개의 심박수 평균을 최종 심박수로 설정
                heart_rate_history.append(final_heart_rate)  # 최종 심박수를 히스토리에 추가
                smoothed_heart_rate = moving_average(heart_rate_history)  # 이동 평균으로 부드러운 심박수 계산

                current_time = time.time()  # 현재 시간 기록
                
                # 마지막 디스플레이 업데이트 시간과 비교하여 일정 간격이 지났는지 확인
                if current_time - last_display_time > update_interval:
                    last_heart_rate = smoothed_heart_rate  # 부드러운 심박수를 최종 심박수로 설정
                    last_display_time = current_time  # 마지막 디스플레이 시간을 현재 시간으로 갱신

                # 실시간 심박수(BPM) 추가 및 10초 평균 계산
                bpm_values.append(last_heart_rate)  # 현재 심박수를 저장
                avg_bpm = np.mean(bpm_values)  # 10초 동안의 평균 심박수 계산
                
                # 실시간 심박수(BPM) 추가 및 5초 평균 계산
                bpm_values_1.append(last_heart_rate)  # 현재 심박수를 별도 저장
                avg_bpm_5 = np.mean(bpm_values_1)  # 5초 동안의 평균 심박수 계산

                display_triangular_regions(frame, regions)  # 얼굴의 ROI 영역을 삼각형으로 시각화하여 프레임에 표시

                # 최종 계산된 심박수와 평균 심박수를 프레임에 텍스트로 표시
                if last_heart_rate is not None:
                    cv2.putText(frame, f'Heart Rate: {int(last_heart_rate)} BPM', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)  # 실시간 심박수
                    cv2.putText(frame, f'10-sec Avg BPM: {int(avg_bpm)}', (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)  # 10초 평균
                    cv2.putText(frame, f'05-sec Avg BPM: {int(avg_bpm_5)}', (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)  # 5초 평균

        # 처리된 프레임을 OpenCV 창에 표시
        cv2.imshow('rPPG Heart Rate Monitor with 5 ROIs', frame)

        # 'q' 키를 누르면 루프 종료
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # 루프 종료 후 자원 해제
    cap.release()  # 웹캠 리소스 해제
    cv2.destroyAllWindows()  # OpenCV 창 닫기

if __name__ == "__main__":
    main()  # 프로그램 실행
